In [ ]:
!pip install ranx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.4/467.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import json
import ast
from ranx import Qrels, Run, evaluate
path = "/content/drive/MyDrive/SE365/Final Project/Eval/Results/"

# Xử lý data

In [ ]:
# Nomalize
def clean_and_parse_list(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value]
    if isinstance(value, str):
        value = value.strip()
        if value.startswith('[') and value.endswith(']') and value != '[]':
            content = value[1:-1]
            if not content.strip():
                return []
            return [item.replace("'", "").replace('"', "").strip() for item in content.split(',')]
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed]
        except:
            pass

    return []

# Xử lý file kết quả json
def preprocess(path: str):
  df = pd.read_json(path)
  df['id'] = range(1, len(df) + 1)
  df = df[['id','ids']]
  df.rename(columns = {'ids':'predict'}, inplace = True)
  df['predict'] = df['predict'].apply(clean_and_parse_list)
  return df

# Eval
def evaluate_retrieval(df_ground_truth, df_predictions):
    qrels_dict = {}
    for _, row in df_ground_truth.iterrows():
        query_id = str(row['id'])
        labels = row['label']
        qrels_dict[query_id] = {str(doc_id): 1 for doc_id in labels}

    run_dict = {}
    for _, row in df_predictions.iterrows():
        query_id = str(row['id'])
        preds = row['predict']
        total_docs = len(preds)
        run_dict[query_id] = {str(doc_id): (total_docs - idx) for idx, doc_id in enumerate(preds)}

    qrels = Qrels(qrels_dict)
    run = Run(run_dict)

    metrics = [
        "hit_rate@1", "hit_rate@5", "hit_rate@10",
        "recall@1", "recall@5", "recall@10",
        "precision@1","precision@5", "precision@10",
        "mrr", "map", "ndcg@10"
    ]

    results = evaluate(qrels, run, metrics)

    print("-" * 45)
    print(f"| {'Kết quả':^41} |")
    print("-" * 45)
    print(f"| {'Chỉ số':<18} | {'Điểm số (Mean)':<20} |")
    print("-" * 45)

    for metric_name in metrics:
        score = results.get(metric_name, 0.0)
        display_name = metric_name.replace("hit_rate", "Hit").capitalize() if "hit_rate" in metric_name else metric_name.upper()
        print(f"| {display_name:<18} | {score:<20.4f} |")

    print("-" * 45)

    return results

## Xử lý tập nhãn thật

In [ ]:
df_gt = pd.read_csv(path + "ground_truth_genAnswers.csv")
df_gt = df_gt[['question','relevant_laws']]
df_gt['id'] = range(1, len(df_gt) + 1)
df_gt = df_gt[['id','relevant_laws']]
df_gt.rename(columns = {'relevant_laws':'label'}, inplace = True)
df_gt['label'] = df_gt['label'].apply(clean_and_parse_list)
df_gt.head()

,id,label
0,1,[52_2010_QH12_D19]
1,2,[53_2014_QH13_D11]
2,3,[45_2019_QH14_D139]
3,4,[57_2014_QH13_D44]
4,5,[144_2021_ND-CP_D5]


# Đánh giá

# Baseline

## Baseline 1 (Only Dense)

In [ ]:
df_dense = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_OnlyDense.json')
dense_score = evaluate_retrieval(df_gt, df_dense)

/usr/local/lib/python3.12/dist-packages/ranx/metrics/hit_rate.py:38: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _hit_rate(qrels[i], run[i], k, rel_lvl)


---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.5780               |
| Hit@5              | 0.8316               |
| Hit@10             | 0.9043               |
| RECALL@1           | 0.5033               |
| RECALL@5           | 0.7604               |
| RECALL@10          | 0.8398               |
| PRECISION@1        | 0.5780               |
| PRECISION@5        | 0.1823               |
| PRECISION@10       | 0.1039               |
| MRR                | 0.6892               |
| MAP                | 0.6341               |
| NDCG@10            | 0.6964               |
---------------------------------------------


## Baseline 2 (Only Sparse)

In [ ]:
df_sparse = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_OnlySparse.json')
sparse_score = evaluate_retrieval(df_gt, df_sparse)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.5337               |
| Hit@5              | 0.7660               |
| Hit@10             | 0.8387               |
| RECALL@1           | 0.4695               |
| RECALL@5           | 0.6926               |
| RECALL@10          | 0.7682               |
| PRECISION@1        | 0.5337               |
| PRECISION@5        | 0.1652               |
| PRECISION@10       | 0.0941               |
| MRR                | 0.6426               |
| MAP                | 0.5876               |
| NDCG@10            | 0.6417               |
---------------------------------------------


# HybridSearch

In [ ]:
df_hs = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_HS.json')
hs_score = evaluate_retrieval(df_gt, df_hs)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.6241               |
| Hit@5              | 0.8670               |
| Hit@10             | 0.9167               |
| RECALL@1           | 0.5450               |
| RECALL@5           | 0.7875               |
| RECALL@10          | 0.8498               |
| PRECISION@1        | 0.6241               |
| PRECISION@5        | 0.1894               |
| PRECISION@10       | 0.1057               |
| MRR                | 0.7272               |
| MAP                | 0.6691               |
| NDCG@10            | 0.7259               |
---------------------------------------------


# HybridSearch + Rewrite

In [ ]:
df_hs_rw = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_HS_RW.json')
hsrw_score = evaluate_retrieval(df_gt, df_hs_rw)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.5940               |
| Hit@5              | 0.8546               |
| Hit@10             | 0.9149               |
| RECALL@1           | 0.5205               |
| RECALL@5           | 0.7781               |
| RECALL@10          | 0.8454               |
| PRECISION@1        | 0.5940               |
| PRECISION@5        | 0.1872               |
| PRECISION@10       | 0.1043               |
| MRR                | 0.7038               |
| MAP                | 0.6483               |
| NDCG@10            | 0.7077               |
---------------------------------------------


# HybridSearch + 2 Rerank

In [ ]:
df_hs_2rr = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_HS_RR.json')
hs2rr_score = evaluate_retrieval(df_gt, df_hs_2rr)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.6188               |
| Hit@5              | 0.8741               |
| Hit@10             | 0.9238               |
| RECALL@1           | 0.5383               |
| RECALL@5           | 0.7961               |
| RECALL@10          | 0.8553               |
| PRECISION@1        | 0.6188               |
| PRECISION@5        | 0.1894               |
| PRECISION@10       | 0.1051               |
| MRR                | 0.7357               |
| MAP                | 0.6757               |
| NDCG@10            | 0.7320               |
---------------------------------------------


# HybridSearch + 1 Rerank

## GTE

In [ ]:
df_hs_rrgte = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_HS_1RRv1.json')
hsrrgte_score = evaluate_retrieval(df_gt, df_hs_rrgte)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.4876               |
| Hit@5              | 0.7872               |
| Hit@10             | 0.8723               |
| RECALL@1           | 0.4214               |
| RECALL@5           | 0.7123               |
| RECALL@10          | 0.8001               |
| PRECISION@1        | 0.4876               |
| PRECISION@5        | 0.1691               |
| PRECISION@10       | 0.0968               |
| MRR                | 0.6158               |
| MAP                | 0.5611               |
| NDCG@10            | 0.6293               |
---------------------------------------------


## BGE

In [ ]:
df_hs_rrbge = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_HS_1RRv3.json')
hsrrbge_score = evaluate_retrieval(df_gt, df_hs_rrbge)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.7394               |
| Hit@5              | 0.9060               |
| Hit@10             | 0.9362               |
| RECALL@1           | 0.6427               |
| RECALL@5           | 0.8245               |
| RECALL@10          | 0.8707               |
| PRECISION@1        | 0.7394               |
| PRECISION@5        | 0.1979               |
| PRECISION@10       | 0.1076               |
| MRR                | 0.8139               |
| MAP                | 0.7446               |
| NDCG@10            | 0.7903               |
---------------------------------------------


# Full pipeline - 2 Rerank

In [ ]:
df_full_2rr = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_Full.json')
full2rr_score = evaluate_retrieval(df_gt, df_full_2rr)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.6241               |
| Hit@5              | 0.8723               |
| Hit@10             | 0.9309               |
| RECALL@1           | 0.5428               |
| RECALL@5           | 0.7943               |
| RECALL@10          | 0.8614               |
| PRECISION@1        | 0.6241               |
| PRECISION@5        | 0.1894               |
| PRECISION@10       | 0.1062               |
| MRR                | 0.7383               |
| MAP                | 0.6773               |
| NDCG@10            | 0.7353               |
---------------------------------------------


# Full pipeline - 1 Rerank

## Rerank v1

In [ ]:
df_full_1rrgte = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_Full_1RRv1.json')
full1rrgte_score = evaluate_retrieval(df_gt, df_full_1rrgte)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.4858               |
| Hit@5              | 0.7926               |
| Hit@10             | 0.8723               |
| RECALL@1           | 0.4196               |
| RECALL@5           | 0.7176               |
| RECALL@10          | 0.8010               |
| PRECISION@1        | 0.4858               |
| PRECISION@5        | 0.1702               |
| PRECISION@10       | 0.0970               |
| MRR                | 0.6153               |
| MAP                | 0.5608               |
| NDCG@10            | 0.6291               |
---------------------------------------------


## Rerank v2

In [ ]:
df_full_1rrbge = preprocess('/content/drive/MyDrive/SE365/Final Project/Eval/Results/final_result_Full_1RRv3.json')
full1rrbge_score = evaluate_retrieval(df_gt, df_full_1rrbge)

---------------------------------------------
|                 Kết quả                 |
---------------------------------------------
| Chỉ số             | Điểm số (Mean)       |
---------------------------------------------
| Hit@1              | 0.7394               |
| Hit@5              | 0.9025               |
| Hit@10             | 0.9362               |
| RECALL@1           | 0.6427               |
| RECALL@5           | 0.8240               |
| RECALL@10          | 0.8725               |
| PRECISION@1        | 0.7394               |
| PRECISION@5        | 0.1975               |
| PRECISION@10       | 0.1076               |
| MRR                | 0.8134               |
| MAP                | 0.7443               |
| NDCG@10            | 0.7904               |
---------------------------------------------


# Tổng hợp kết quả

In [ ]:
results_summary = {
    "Baseline 1 (Dense)": dense_score,
    "Baseline 2 (Sparse)": sparse_score,
    "HS": hs_score,
    "HS + RW": hsrw_score,
    "HS + 2 RR": hs2rr_score,
    "HS + RR_v1": hsrrgte_score,
    "HS + RR_v3": hsrrbge_score,
    "HS + RW + 2 RR": full2rr_score,
    "HS + RW + RR_v1": full1rrgte_score,
    "HS + RW + RR_v3": full1rrbge_score,
}

df_summary = pd.DataFrame(results_summary)
df_summary.index.name = "Metric"
df_summary.index = df_summary.index.map(lambda x: x.replace('hit_rate', 'Hit_Rate').replace('recall', 'Recall').replace('precision', 'Precision').upper())

display(df_summary)

,Baseline 1 (Dense),Baseline 2 (Sparse),HS,HS + RW,HS + 2 RR,HS + RR_v1,HS + RR_v3,HS + RW + 2 RR,HS + RW + RR_v1,HS + RW + RR_v3
Metric,,,,,,,,,,
HIT_RATE@1,0.578014,0.533688,0.624113,0.593972,0.618794,0.487589,0.739362,0.624113,0.485816,0.739362
HIT_RATE@5,0.831560,0.765957,0.867021,0.854610,0.874113,0.787234,0.906028,0.872340,0.792553,0.902482
HIT_RATE@10,0.904255,0.838652,0.916667,0.914894,0.923759,0.872340,0.936170,0.930851,0.872340,0.936170
RECALL@1,0.503324,0.469489,0.544991,0.520464,0.538342,0.421395,0.642657,0.542775,0.419622,0.642657
RECALL@5,0.760417,0.692598,0.787530,0.778073,0.796099,0.712323,0.824542,0.794326,0.717642,0.823951
RECALL@10,0.839761,0.768248,0.849808,0.845375,0.855349,0.800089,0.870715,0.861407,0.800975,0.872488
PRECISION@1,0.578014,0.533688,0.624113,0.593972,0.618794,0.487589,0.739362,0.624113,0.485816,0.739362
PRECISION@5,0.182270,0.165248,0.189362,0.187234,0.189362,0.169149,0.197872,0.189362,0.170213,0.197518
PRECISION@10,0.103901,0.094149,0.105674,0.104255,0.105142,0.096809,0.107624,0.106206,0.096986,0.107624


In [ ]:
df_summary_transposed = df_summary.T
display(df_summary_transposed.style.format('{:.4f}'))

Metric,HIT_RATE@1,HIT_RATE@5,HIT_RATE@10,RECALL@1,RECALL@5,RECALL@10,PRECISION@1,PRECISION@5,PRECISION@10,MRR,MAP,NDCG@10
Baseline 1 (Dense),0.5780,0.8316,0.9043,0.5033,0.7604,0.8398,0.5780,0.1823,0.1039,0.6892,0.6341,0.6964
Baseline 2 (Sparse),0.5337,0.7660,0.8387,0.4695,0.6926,0.7682,0.5337,0.1652,0.0941,0.6426,0.5876,0.6417
HS,0.6241,0.8670,0.9167,0.5450,0.7875,0.8498,0.6241,0.1894,0.1057,0.7272,0.6691,0.7259
HS + RW,0.5940,0.8546,0.9149,0.5205,0.7781,0.8454,0.5940,0.1872,0.1043,0.7038,0.6483,0.7077
HS + 2 RR,0.6188,0.8741,0.9238,0.5383,0.7961,0.8553,0.6188,0.1894,0.1051,0.7357,0.6757,0.7320
HS + RR_v1,0.4876,0.7872,0.8723,0.4214,0.7123,0.8001,0.4876,0.1691,0.0968,0.6158,0.5611,0.6293
HS + RR_v3,0.7394,0.9060,0.9362,0.6427,0.8245,0.8707,0.7394,0.1979,0.1076,0.8139,0.7446,0.7903
HS + RW + 2 RR,0.6241,0.8723,0.9309,0.5428,0.7943,0.8614,0.6241,0.1894,0.1062,0.7383,0.6773,0.7353
HS + RW + RR_v1,0.4858,0.7926,0.8723,0.4196,0.7176,0.8010,0.4858,0.1702,0.0970,0.6153,0.5608,0.6291
HS + RW + RR_v3,0.7394,0.9025,0.9362,0.6427,0.8240,0.8725,0.7394,0.1975,0.1076,0.8134,0.7443,0.7904
